In [7]:
from transformers import AutoTokenizer, AutoModel
from datasets import load_dataset
import torch
import gensim.downloader
import pandas as pd
import spacy
from collections import defaultdict
import random
import json

In [8]:
dataset = load_dataset("wikitext", "wikitext-103-v1", split="train")
#glove = gensim.downloader.load("glove-wiki-gigaword-50")

In [9]:
df = pd.read_csv("/home/onyxia/work/morph_families.tsv", 
                sep="\t", 
                names=["lemma", "forms", "fam", "transf"])
                
#glove_voc = set(glove.key_to_index.keys())
#print(glove_voc)


In [10]:
content = []
for line in dataset["text"]:
    line = line.strip().lower()
    if line and not line.startswith("=") and len(line.split()) > 4:
        content.append(line)


In [11]:
target = {word for form in df["forms"].str.split(",") for word in form}
del df
del dataset

In [12]:
nlp = spacy.blank("en")
nlp.add_pipe("sentencizer")

matched = defaultdict(list)
sentences =  []

for doc in nlp.pipe(content, batch_size=2048, n_process=2):
    for sent in doc.sents:
        if len(sent) > 40:
            continue

        sent_text = sent.text
        tokens = {token.text for token in sent}

        for word in target.intersection(tokens):
            matched[word].append(sent_text)

In [13]:
def tag_word(sents):
    n = len(sents)
    if n >= 50:
        return "sampled"
    elif n >= 10:
        return "low_freq"
    return "excluded"

def sample_sents(row):
    if row["tag"] == "sampled":
        return random.sample(row["sents"], 50)
    return row["sents"]

print("sent_df")
random.seed(42)

sent_df = pd.DataFrame(matched.items(),
                        columns=["word", "sents"])

sent_df["tag"] = sent_df["sents"].apply(tag_word)
sent_df["sents"] = sent_df.apply(sample_sents, axis="columns")


sent_df.to_json("sent_df.json")

sent_df


In [14]:
sent_df.to_json("sent_df.json")